In [2]:
# ===================== 跨日期 RF1DCNN SNR Sweep =====================
# 训练集：一次性固定噪声 | 验证集：干净 | 测试集：指定SNR噪声
from joblib import load
import pandas as pd
import numpy as np
import os
from data_utilities import *
import cv2
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import gc
from tqdm.auto import tqdm
from collections import defaultdict

import torch
torch._dynamo.disable()
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
import seaborn as sns
from datetime import datetime

# ===================== 数据集加载 =====================
dataset_name = 'ManySig'
dataset_path='../ManySig.pkl/'
compact_dataset = load_compact_pkl_dataset(dataset_path,dataset_name)

tx_list = compact_dataset['tx_list']
rx_list = compact_dataset['rx_list']
capture_date_list = compact_dataset['capture_date_list']

n_tx = len(tx_list)
n_rx = len(rx_list)
print(f"TX: {n_tx}, RX: {n_rx}, Dates: {len(capture_date_list)}")

# ===================== 参数设置 =====================
equalized = 0
max_sig = None
block_size = 240
y = 5

train_dates = ['2021_03_15']
test_dates  = ['2021_03_01']

X_train, y_train, X_test, y_test = preprocess_dataset_cross_IQ_blocks_single_date_per_rx_cyclic(
    compact_dataset=compact_dataset,
    tx_list=tx_list,
    train_dates=train_dates,
    test_dates=test_dates,
    max_sig=max_sig,
    equalized=equalized,
    block_size=block_size,
    y=y
)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

# ===================== 数据增强相关 =====================
FS = 20e6
FC = 2.4e9
VELOCITY_KMH = 120

def compute_doppler_shift(v_kmh, fc_hz):
    if not v_kmh:
        return 0.0
    c = 3e8
    v = v_kmh / 3.6
    return fc_hz * v / c

def add_complex_awgn(signal, snr_db):
    if snr_db is None:
        return signal
    power = np.mean(np.abs(signal)**2)
    noise_power = power / (10**(snr_db/10))
    noise_std = np.sqrt(noise_power/2)
    noise = noise_std * (np.random.randn(*signal.shape) + 1j*np.random.randn(*signal.shape))
    return signal + noise

def apply_doppler_shift(signal, fd_hz, fs_hz):
    if fd_hz is None or fd_hz == 0:
        return signal
    t = np.arange(len(signal)) / fs_hz
    return signal * np.exp(1j * 2 * np.pi * fd_hz * t)

# ===================== 静态预处理函数 =====================
def preprocess_iq_data_static(data_real_imag, add_noise=False, snr_db=None, 
                                 add_doppler=False, fd_hz=None, fs_hz=FS):
    """
    统一的预处理：先标准化，再加噪声
    """
    data = data_real_imag.astype(np.float32).copy()
    N, L, C = data.shape
    out = np.empty_like(data, dtype=np.float32)
    
    for i in range(N):
        iq = data[i]
        
        # 1. 先对干净数据标准化
        mu_clean = iq.mean(axis=0)
        sigma_clean = iq.std(axis=0)
        sigma_clean[sigma_clean < 1e-8] = 1.0
        iq_normalized_clean = (iq - mu_clean) / sigma_clean
        
        # 2. 转换为复数
        sig_complex = iq_normalized_clean[..., 0] + 1j * iq_normalized_clean[..., 1]
        
        # 3. 加噪声（在标准化后的数据上加噪声）
        if add_noise and snr_db is not None:
            sig_complex = add_complex_awgn(sig_complex, snr_db)
        
        # 4. 加多普勒
        if add_doppler and fd_hz is not None:
            sig_complex = apply_doppler_shift(sig_complex, fd_hz, fs_hz)
        
        # 5. 转换回实数并再次标准化（轻微调整）
        iq_processed = np.stack([np.real(sig_complex), np.imag(sig_complex)], axis=-1)
        mu_final = iq_processed.mean(axis=0)
        sigma_final = iq_processed.std(axis=0)
        sigma_final[sigma_final < 1e-8] = 1.0
        iq_final = (iq_processed - mu_final) / sigma_final
        
        out[i] = iq_final
    
    return torch.tensor(out, dtype=torch.float32)

# ===================== RF1DCNN 模型 =====================
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, 1, padding, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = None
        if in_channels != out_channels or stride != 1:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        identity = x
        out = self.conv1(x); out = self.bn1(out); out = self.relu(out)
        out = self.conv2(out); out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(identity)
        out += identity
        out = self.relu(out)
        return out

class RF1DCNN(nn.Module):
    def __init__(self, num_classes, dropout_conv=0.1, dropout_fc=0.3):
        super().__init__()
        self.layer1 = nn.Sequential(ResidualBlock1D(2, 32, kernel_size=7), nn.Dropout(dropout_conv))
        self.pool1 = nn.MaxPool1d(2)
        self.layer2 = nn.Sequential(ResidualBlock1D(32, 64, kernel_size=5), nn.Dropout(dropout_conv))
        self.pool2 = nn.MaxPool1d(2)
        self.layer3 = nn.Sequential(ResidualBlock1D(64, 128, kernel_size=5), nn.Dropout(dropout_conv))
        self.pool3 = nn.MaxPool1d(2)
        self.layer4 = nn.Sequential(ResidualBlock1D(128, 256, kernel_size=3), nn.Dropout(dropout_conv))
        self.pool4 = nn.MaxPool1d(2)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 15, 512),
            nn.ReLU(),
            nn.Dropout(dropout_fc),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = x.permute(0,2,1)
        x = self.layer1(x); x = self.pool1(x)
        x = self.layer2(x); x = self.pool2(x)
        x = self.layer3(x); x = self.pool3(x)
        x = self.layer4(x); x = self.pool4(x)
        return self.fc(x)

# ===================== 辅助函数 =====================
def evaluate_model(model, dataloader, device, num_classes):
    model.eval()
    correct, total = 0, 0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            _, p = torch.max(out, 1)
            correct += (p == yb).sum().item()
            total += yb.size(0)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(p.cpu().numpy())
    acc = 100.0 * correct / total if total>0 else 0.0
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    return acc, cm

def plot_confusion_matrix(cm, classes, fold, save_folder, dataset_type='Test'):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{dataset_type} Confusion Matrix Fold{fold}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.savefig(os.path.join(save_folder, f'{dataset_type.lower()}_cm_fold{fold}.png')); plt.close()

def plot_curves(train_losses, val_losses, train_accs, val_accs, fold, save_folder):
    plt.figure()
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'Fold {fold} Loss')
    plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(save_folder, f'loss_fold{fold}.png')); plt.close()
    
    plt.figure()
    plt.plot(train_accs, label='Train Acc')
    plt.plot(val_accs, label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title(f'Fold {fold} Accuracy')
    plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(save_folder, f'acc_fold{fold}.png')); plt.close()

# ===================== K-Fold 训练（修改版） =====================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_ROOT = "./training_results"
os.makedirs(SAVE_ROOT, exist_ok=True)
BATCH_SIZE = 256
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-4
N_SPLITS = 5
PATIENCE = 8
MIN_DELTA = 0.01

# ===================== 修复数据泄露的版本 =====================
def train_kfold_pointcloud(X_train, y_train, X_test, y_test, num_classes,
                                device=DEVICE, batch_size=BATCH_SIZE, epochs=EPOCHS,
                                lr=LR, weight_decay=WEIGHT_DECAY,
                                n_splits=N_SPLITS, patience=PATIENCE, min_delta=MIN_DELTA,
                                snr_db=None, add_noise=True, add_doppler=False):
    
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    save_dir = f"{timestamp}_wisig_cross_SNR{snr_db}dB_classes{num_classes}_ResNet_fixed"
    save_folder = os.path.join(SAVE_ROOT, save_dir)
    os.makedirs(save_folder, exist_ok=True)
    results_file = os.path.join(save_folder, "results.txt")

    with open(results_file, 'w') as f:
        f.write("=== Experiment Parameters (Fixed Data Leakage) ===\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"SNR_dB: {snr_db}, Training: Static noise | Validation: Clean | Test: Noisy\n")
        f.write(f"Data Split: No leakage - train/val samples are completely disjoint\n")
        f.write("============================\n\n")

    # 计算多普勒频移
    fd = int(compute_doppler_shift(VELOCITY_KMH, FC)) if add_doppler else None
    
    # 预处理测试集（一次性）
    print("预处理测试集...")
    X_test_noisy = preprocess_iq_data_static(
        X_test, add_noise=True, snr_db=snr_db, 
        add_doppler=add_doppler, fd_hz=fd, fs_hz=FS
    )
    X_test_clean = preprocess_iq_data_static(X_test, add_noise=False)
    
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)
    test_dataset_noisy = TensorDataset(X_test_noisy, y_test_tensor)
    test_dataset_clean = TensorDataset(X_test_clean, y_test_tensor)

    # K-Fold交叉验证（修复数据泄露）
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(X_train))

    fold_results = []
    test_results_noisy = []
    test_results_clean = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(indices)):
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        print(f"训练样本数: {len(tr_idx)}, 验证样本数: {len(va_idx)}")
        
        # 🔴 关键修复：确保训练集和验证集样本不重叠
        X_train_fold = X_train[tr_idx]  # 训练样本
        X_val_fold = X_train[va_idx]    # 验证样本（与训练完全不同）
        y_train_fold = y_train[tr_idx]
        y_val_fold = y_train[va_idx]
        
        # 训练集：添加固定噪声
        print("处理训练集（添加噪声）...")
        X_train_noisy = preprocess_iq_data_static(
            X_train_fold, add_noise=add_noise, snr_db=snr_db,
            add_doppler=add_doppler, fd_hz=fd, fs_hz=FS
        )
        
        # 验证集：保持干净
        print("处理验证集（保持干净）...")
        X_val_clean = preprocess_iq_data_static(X_val_fold, add_noise=False)
        
        # 创建数据集
        train_dataset = TensorDataset(X_train_noisy, torch.tensor(y_train_fold, dtype=torch.long))
        val_dataset = TensorDataset(X_val_clean, torch.tensor(y_val_fold, dtype=torch.long))
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
        test_loader_noisy = DataLoader(test_dataset_noisy, batch_size=batch_size, shuffle=False)
        test_loader_clean = DataLoader(test_dataset_clean, batch_size=batch_size, shuffle=False)

        # 模型训练（其余代码保持不变）
        model = RF1DCNN(num_classes=num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

        best_train_loss = float('inf')
        best_val_loss = float('inf')
        best_weights = None
        patience_cnt = 0
        train_losses, val_losses, train_accs, val_accs = [], [], [], []
        
        for epoch in range(epochs):
            # === 训练阶段 ===
            model.train()
            running_loss, correct, total = 0.0, 0, 0
            
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                outputs = model(xb)
                loss = criterion(outputs, yb)
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == yb).sum().item()
                total += yb.size(0)
            
            train_loss = running_loss / len(train_loader)
            train_acc = 100.0 * correct / total
            train_losses.append(train_loss)
            train_accs.append(train_acc)
            
            # === 验证阶段 ===
            model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    outputs = model(xb)
                    loss = criterion(outputs, yb)
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs, 1)
                    val_correct += (predicted == yb).sum().item()
                    val_total += yb.size(0)
            
            val_loss /= len(val_loader)
            val_acc = 100.0 * val_correct / val_total
            val_losses.append(val_loss)
            val_accs.append(val_acc)
            
            scheduler.step(val_loss)
            
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"TrainLoss: {train_loss:.4f} | ValLoss: {val_loss:.4f} | "
                  f"TrainAcc: {train_acc:.2f}% | ValAcc: {val_acc:.2f}%")
            
            # === 改进的早停策略 ===
            # 方案1：基于训练损失早停（简单有效）
            if train_loss < best_train_loss - min_delta:
                best_train_loss = train_loss
                best_val_loss = val_loss
                best_weights = model.state_dict().copy()
                patience_cnt = 0
                print(f"✅ 训练损失改善: {best_train_loss:.4f}")
            else:
                patience_cnt += 1
                print(f"⏳ 训练损失停滞: {patience_cnt}/{patience}")
                if patience_cnt >= patience:
                    print(f"🛑 早停触发 (训练损失 {patience_cnt} 个epoch未改善)")
                    break
            
        
        # 加载最佳模型并评估
        if best_weights:
            model.load_state_dict(best_weights)
        
        train_acc, train_cm = evaluate_model(model, train_loader, device, num_classes)
        val_acc, val_cm = evaluate_model(model, val_loader, device, num_classes)
        test_acc_noisy, test_cm_noisy = evaluate_model(model, test_loader_noisy, device, num_classes)
        test_acc_clean, test_cm_clean = evaluate_model(model, test_loader_clean, device, num_classes)
        
        fold_results.append(val_acc)
        test_results_noisy.append(test_acc_noisy)
        test_results_clean.append(test_acc_clean)
        
        print(f"Fold {fold+1} Results:")
        print(f"  Validation (Clean): {val_acc:.2f}%")
        print(f"  Test (Noisy): {test_acc_noisy:.2f}%")
        print(f"  Test (Clean): {test_acc_clean:.2f}%")
    
    # 最终总结
    avg_val = np.mean(fold_results)
    avg_test_noisy = np.mean(test_results_noisy)
    avg_test_clean = np.mean(test_results_clean)
    
    print(f"\n=== Final Summary (No Data Leakage) ===")
    print(f"Validation Accuracy: {avg_val:.2f}% ± {np.std(fold_results):.2f}%")
    print(f"Test Accuracy (Noisy): {avg_test_noisy:.2f}% ± {np.std(test_results_noisy):.2f}%")
    print(f"Test Accuracy (Clean): {avg_test_clean:.2f}% ± {np.std(test_results_clean):.2f}%")
    
    return save_folder

# ===================== SNR Sweep =====================
def run_experiment_with_snr(X_train, y_train, X_test, y_test, snr_db):
    print("\n" + "="*50)
    print(f"🚀 Running SNR = {snr_db} dB")
    print("="*50)
    save_folder = train_kfold_pointcloud(
        X_train, y_train, X_test, y_test, 
        num_classes=len(np.unique(y_train)),
        snr_db=snr_db, add_noise=True, add_doppler=True
    )
    print(f"🎉 Finished SNR={snr_db} dB, results in: {save_folder}")
    return save_folder

# ===================== 批量 SNR Sweep =====================
snr_list = list(range(-10, -41, -10))
all_results = {}
for snr in snr_list:
    folder = run_experiment_with_snr(X_train, y_train, X_test, y_test, snr)
    all_results[snr] = folder

print("\n\n================ FINAL SUMMARY ================")
for snr, folder in all_results.items():
    print(f"SNR {snr:>3} dB → results in: {folder}")
print("==============================================")

TX: 6, RX: 12, Dates: 4
X_train shape: (76800, 240, 2)
X_test  shape: (76800, 240, 2)

🚀 Running SNR = -30 dB
预处理测试集...

=== Fold 1/5 ===
训练样本数: 61440, 验证样本数: 15360


KeyboardInterrupt: 